<a href="https://colab.research.google.com/github/kkhushikinger/Small_Projects/blob/main/movie_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this first in Colab (comment out locally if already installed)
!pip install scikit-surprise pandas scikit-learn tqdm --quiet
print("Installed required packages.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 10.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installed required packages.


In [2]:
!pip install scikit-learn pandas numpy


In [3]:
!wget -nc https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip


--2026-02-25 08:32:33--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  2.78MB/s    in 1.7s    

2026-02-25 08:32:36 (2.78 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.ba

In [4]:
import pandas as pd

ratings = pd.read_csv("ml-100k/u.data", sep="\t", names=["user_id","movie_id","rating","timestamp"])
movies = pd.read_csv("ml-100k/u.item", sep="|", header=None, encoding="latin-1")

movies = movies[[0,1]]   # movie_id and title only
movies.columns = ["movie_id", "title"]

df = ratings.merge(movies, on="movie_id")
df.head()



,user_id,movie_id,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [5]:
import numpy as np

user_item_matrix = df.pivot_table(index="user_id", columns="movie_id", values="rating").fillna(0)
user_item_matrix.head()


movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=20, random_state=42)
matrix_svd = svd.fit_transform(user_item_matrix)

matrix_svd.shape


(943, 20)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity,
                                  index=user_item_matrix.columns,
                                  columns=user_item_matrix.columns)


In [8]:
def recommend_movies(user_id, num_recommend=10):
    user_ratings = user_item_matrix.loc[user_id]
    watched_movies = user_ratings[user_ratings > 0].index.tolist()

    scores = {}

    for movie in watched_movies:
        similar_movies = item_similarity_df[movie].sort_values(ascending=False)[1:11]

        for sim_movie, score in similar_movies.items():
            if sim_movie not in watched_movies:
                scores[sim_movie] = scores.get(sim_movie, 0) + score

    recommended = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:num_recommend]

    result = pd.DataFrame(recommended, columns=["movie_id", "score"])
    return result.merge(movies, on="movie_id")


In [9]:
recommend_movies(10, 10)
# This line calls the recommend_movies function for user ID 10
# and returns the top 10 movie recommendations for that user

,movie_id,score,title
0,204,19.836090,Back to the Future (1985)
1,172,12.623680,"Empire Strikes Back, The (1980)"
2,79,9.764448,"Fugitive, The (1993)"
3,423,9.469824,E.T. the Extra-Terrestrial (1982)
4,210,7.957008,Indiana Jones and the Last Crusade (1989)
5,96,7.373123,Terminator 2: Judgment Day (1991)
6,89,7.099320,Blade Runner (1982)
7,196,6.639558,Dead Poets Society (1989)
8,202,6.208729,Groundhog Day (1993)
9,208,5.812970,Young Frankenstein (1974)
